In [21]:
import sys
sys.path.append('photon')

In [22]:
%load_ext autoreload
%autoreload 2

import torch
from scipy import ndimage
import torch.nn.functional as F
from data import BeamData, Plan
import torch.nn as nn
from pathlib import Path
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy import ndimage
from mlp import MLPProcessor, DoseModel
import torch.optim as optim


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
data_dir = Path("/workspace/DoseRAD2026Dev/data/DoseRAD2026/photon/training")

np.random.seed(1234)
pt_list = [f.name for f in data_dir.iterdir() if f.is_dir()]
np.random.shuffle(pt_list)

pt_tr = pt_list[:50]
pt_vl = pt_list[50:60]
pt_ts = pt_list[60:]

In [4]:
for pt in pt_tr:
    pat_dir = data_dir / pt
    plan = Plan(
        img_file_path=rf"{pat_dir}/image/ct.mha",
        info_json_path=rf"{pat_dir}/{pt}.json",
        dose_dir=rf"{pat_dir}/dose",
    )
    print(pt)

    break

State set to 0
1ABB143


In [5]:
d = BeamData(plan, 30)

Reading dose files: 100%|██████████| 30/30 [00:01<00:00, 26.96it/s]


Starting parallel load with 8 threads...

Finished. Final array shape: (30, 129, 252, 276)
Data type: float16


Rotating dose & img: 100%|██████████| 1/1 [00:02<00:00,  2.86s/it]


Grid created


Calulating BEV beam path: 100%|██████████| 1/1 [00:00<00:00,  1.11it/s]


In [7]:
d.plan.isocentre_ijk

(129, 84, 76)

In [7]:
img, bev, mask, loc, dose = d[0]

In [ ]:
model = DoseModel()


In [30]:
optimizers = [
    optim.Adam(getattr(model, key).parameters(), lr=0.001) for key in ['bev3', 'ring3', 'ring2', 'ring1']
]
l1 = nn.L1Loss()
l2 = nn.MSELoss()

In [36]:
print("Starting MLP training...")

n = 0
for pt_id, pt in enumerate(pt_tr):
    n += 1
    pat_dir = data_dir / pt
    plan = Plan(
        img_file_path=rf"{pat_dir}/image/ct.mha",
        info_json_path=rf"{pat_dir}/{pt}.json",
        dose_dir=rf"{pat_dir}/dose",
    )
    d = BeamData(plan, 5)
    for img, bev, mask, loc, dose in d:
        proc = MLPProcessor(img, bev, dose)

        epochs = 50
        pbar = tqdm(range(epochs))
        for epoch in pbar:
            # Forward pass
            xs, ys = proc.get_xy()

            preds = model(xs)
            weights = [3,1,1,1]

            ls = [l1(pred, y)*w for (pred,y,w) in zip(preds,ys,weights)]
            for o, l in zip(optimizers, ls):
                o.zero_grad()
                l.backward()
                o.step()


            pbar.set_postfix_str(f'Epoch [{epoch+1}/{epochs}], Loss: {torch.stack(ls).tolist()}')

    # Save weights
    torch.save(model.bev3.state_dict(), f'mlp_weights/{n}-bev3-{pt_id}.pth')
    torch.save(model.ring3.state_dict(), f'mlp_weights/{n}-ring3-{pt_id}.pth')
    torch.save(model.ring2.state_dict(), f'mlp_weights/{n}-ring2-{pt_id}.pth')
    torch.save(model.ring1.state_dict(), f'mlp_weights/{n}-ring1-{pt_id}.pth')

Starting MLP training...
State set to 0


Reading dose files: 100%|██████████| 5/5 [00:00<00:00, 19.01it/s]


Starting parallel load with 8 threads...

Finished. Final array shape: (5, 129, 252, 276)
Data type: float16


Rotating dose & img: 100%|██████████| 1/1 [00:00<00:00,  2.42it/s]


Grid created


...

Starting parallel load with 8 threads...

Finished. Final array shape: (5, 200, 252, 274)
Data type: float16


Rotating dose & img: 100%|██████████| 1/1 [00:00<00:00,  1.64it/s]


Grid created


 90%|█████████ | 45/50 [00:07<00:00,  5.99it/s, Epoch [45/50], Loss: [0.45084816217422485, 0.04917694628238678, 0.07489856332540512, 0.12853774428367615]]  


KeyboardInterrupt: 